# Phase III Full Serialization Pipeline

This notebook processes all patients through all 16 experimental conditions and saves the serialized prompts to organized folders for later LLM processing.

## Output Structure:
```
serialized_prompts/
├── PA_FF1/
│   ├── train/
│   │   ├── patient_000001.txt
│   │   ├── patient_000002.txt
│   │   └── ...
│   ├── val/
│   │   ├── patient_014826.txt
│   │   └── ...
│   └── test/
│       ├── patient_016944.txt
│       └── ...
├── PA_FF2/
│   ├── train/
│   ├── val/
│   └── test/
└── PB_FF8/
    ├── train/
    ├── val/
    └── test/
```


In [1]:
import pandas as pd
import numpy as np
import pickle
import json
from pathlib import Path
from tqdm import tqdm
import sys
import warnings
warnings.filterwarnings('ignore')

# Import Phase 3 components
from phase3_components import ReferenceRangeManager, FormatGenerator, get_all_format_ids
from phase3_prompts import PromptManager, get_experimental_conditions

print("Imports complete")


Imports complete


## Setup Output Directories


In [ ]:
# Create output directory structure with split subdirectories
output_base = Path('../../serialized_prompts')
output_base.mkdir(exist_ok=True)

# Get all experimental conditions
conditions = get_experimental_conditions()

# Define splits
splits = ['train', 'val', 'test']

# Create subdirectories for each condition and split
for condition in conditions:
    condition_dir = output_base / condition['condition_name']
    condition_dir.mkdir(exist_ok=True)
    
    # Create split subdirectories within each condition
    for split in splits:
        split_dir = condition_dir / split
        split_dir.mkdir(exist_ok=True)
    
print(f"Created output directories for {len(conditions)} conditions with {len(splits)} splits each in: {output_base}")
print(f"Conditions: {[c['condition_name'] for c in conditions]}")
print(f"Splits: {splits}")

# Example final structure:
print(f"\nDirectory structure example:")
print(f"  {output_base}/PA_FF1/train/")
print(f"  {output_base}/PA_FF1/val/") 
print(f"  {output_base}/PA_FF1/test/")
print(f"  {output_base}/PB_FF8/train/")
print(f"  ... (etc)")


In [2]:
# Create output directory structure
output_base = Path('../../serialized_prompts')
output_base.mkdir(exist_ok=True)

# Get all experimental conditions
conditions = get_experimental_conditions()

# Create subdirectories for each condition
for condition in conditions:
    condition_dir = output_base / condition['condition_name']
    condition_dir.mkdir(exist_ok=True)
    
print(f"Created output directories for {len(conditions)} conditions in: {output_base}")
print(f"Conditions: {[c['condition_name'] for c in conditions]}")


Created output directories for 16 conditions in: ../../serialized_prompts
Conditions: ['PA_FF1', 'PA_FF2', 'PA_FF3', 'PA_FF4', 'PA_FF5', 'PA_FF6', 'PA_FF7', 'PA_FF8', 'PB_FF1', 'PB_FF2', 'PB_FF3', 'PB_FF4', 'PB_FF5', 'PB_FF6', 'PB_FF7', 'PB_FF8']


## Load Phase I Data


In [3]:
phase1_dir = Path('../Phase 1/phase_1_outputs')

def load_pickle_safe(filepath):
    """Handle pandas version compatibility for pickle loading"""
    try:
        with open(filepath, 'rb') as f:
            return pickle.load(f)
    except ModuleNotFoundError:
        # Handle pandas version compatibility
        class CompatibilityModule:
            def __getattr__(self, name):
                return pd.Index if name.endswith('Index') else pd.RangeIndex
        
        sys.modules['pandas.core.indexes.numeric'] = CompatibilityModule()
        try:
            with open(filepath, 'rb') as f:
                return pickle.load(f)
        finally:
            if 'pandas.core.indexes.numeric' in sys.modules:
                del sys.modules['pandas.core.indexes.numeric']

# Load preprocessing objects
print("Loading preprocessing objects...")
label_encoders = load_pickle_safe(phase1_dir / 'preprocessed_mort_hosp_trends_True_window_24_gap_6_seed_42_label_encoders.pkl')
scaler = load_pickle_safe(phase1_dir / 'preprocessed_mort_hosp_trends_True_window_24_gap_6_seed_42_scaler.pkl')

# Load all datasets
print("Loading datasets...")
X_train = load_pickle_safe(phase1_dir / 'preprocessed_mort_hosp_trends_True_window_24_gap_6_seed_42_X_train.pkl')
X_val = load_pickle_safe(phase1_dir / 'preprocessed_mort_hosp_trends_True_window_24_gap_6_seed_42_X_val.pkl')
X_test = load_pickle_safe(phase1_dir / 'preprocessed_mort_hosp_trends_True_window_24_gap_6_seed_42_X_test.pkl')

print(f"Dataset shapes:")
print(f"  Train: {X_train.shape}")
print(f"  Validation: {X_val.shape}")
print(f"  Test: {X_test.shape}")
print(f"  Total patients: {X_train.shape[0] + X_val.shape[0] + X_test.shape[0]}")


Loading preprocessing objects...
Loading datasets...
Dataset shapes:
  Train: (14825, 458)
  Validation: (2118, 458)
  Test: (5648, 458)
  Total patients: 22591


## Prepare Full Dataset


In [4]:
# Combine all datasets
all_data = pd.concat([
    X_train.assign(split='train'),
    X_val.assign(split='val'),
    X_test.assign(split='test')
], ignore_index=True)

print(f"Combined dataset shape: {all_data.shape}")

# Unnormalize the scaled features
feature_names = scaler.feature_names_in_
scaled_features = [col for col in all_data.columns if col in feature_names]

print("Unnormalizing scaled features...")
unnormalized_data = all_data.copy()
unnormalized_data[scaled_features] = scaler.inverse_transform(all_data[scaled_features])

# Convert categorical variables back to original values
print("Converting categorical variables...")
for cat_var, encoder in label_encoders.items():
    if cat_var in unnormalized_data.columns:
        try:
            original_values = encoder.inverse_transform(unnormalized_data[cat_var].astype(int))
            unnormalized_data[cat_var] = original_values
        except:
            pass  # Keep as is if inverse transform fails

# Add patient IDs and assign genders (simplified for demo)
unnormalized_data['patient_id'] = [f'patient_{i+1:06d}' for i in range(len(unnormalized_data))]
# Assign gender based on encoded gender if available, otherwise random
if 'gender_encoded' in unnormalized_data.columns:
    unnormalized_data['gender'] = unnormalized_data['gender_encoded'].map({'F': 'female', 'M': 'male'}).fillna('unknown')
else:
    # Random assignment for demo
    np.random.seed(42)
    unnormalized_data['gender'] = np.random.choice(['male', 'female'], size=len(unnormalized_data))

# Define feature columns (exclude metadata)
feature_cols = [col for col in unnormalized_data.columns if col not in ['patient_id', 'gender', 'split']]

print(f"Feature columns: {len(feature_cols)}")
print(f"Sample patient IDs: {unnormalized_data['patient_id'].head().tolist()}")


Combined dataset shape: (22591, 459)
Unnormalizing scaled features...
Converting categorical variables...
Feature columns: 458
Sample patient IDs: ['patient_000001', 'patient_000002', 'patient_000003', 'patient_000004', 'patient_000005']


## Initialize Experiment Components


In [5]:
# Initialize reference range manager
try:
    ref_manager = ReferenceRangeManager("../../data/Lab_reference_ranges.csv")
    print("Loaded reference ranges from file")
except FileNotFoundError:
    print("Reference ranges file not found. Using dummy reference manager.")
    class DummyReferenceManager:
        def get_interpretation(self, feature_name, value, gender='unknown'):
            return 'Normal'  # Placeholder
    ref_manager = DummyReferenceManager()

# Initialize components
format_generator = FormatGenerator(ref_manager)
prompt_manager = PromptManager()

print(f"Initialized components:")
print(f"  Format IDs: {get_all_format_ids()}")
print(f"  Prompt IDs: {prompt_manager.get_all_prompt_ids()}")
print(f"  Total conditions: {len(conditions)}")


Loaded reference ranges from file
Initialized components:
  Format IDs: ['F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8']
  Prompt IDs: ['A', 'B']
  Total conditions: 16


## Process All Patients


In [ ]:
# Process patients in batches for memory efficiency
BATCH_SIZE = 100  # Process 100 patients at a time
total_patients = len(unnormalized_data)
total_outputs = total_patients * len(conditions)

print(f"Processing {total_patients} patients across {len(conditions)} conditions")
print(f"Total outputs to generate: {total_outputs:,}")
print(f"Processing in batches of {BATCH_SIZE}")

# Track progress and errors
processed_count = 0
error_count = 0
error_log = []

# Process in batches
for batch_start in tqdm(range(0, total_patients, BATCH_SIZE), desc="Processing batches"):
    batch_end = min(batch_start + BATCH_SIZE, total_patients)
    batch_data = unnormalized_data.iloc[batch_start:batch_end]
    
    # Process each patient in the batch
    for idx, (_, patient_row) in enumerate(batch_data.iterrows()):
        patient_id = patient_row['patient_id']
        gender = patient_row['gender']
        patient_split = patient_row['split']  # Get the dataset split for this patient
        patient_features = patient_row[feature_cols]
        
        # Process through all conditions
        for condition in conditions:
            try:
                # Generate serialized format
                serialized_data = format_generator.generate_format(
                    patient_data=patient_features,
                    format_id=condition['format_id'],
                    gender=gender
                )
                
                # Combine with prompt
                full_prompt = prompt_manager.get_full_prompt(
                    prompt_id=condition['prompt_id'],
                    serialized_data=serialized_data
                )
                
                # Save to file with split-specific subdirectory
                output_file = output_base / condition['condition_name'] / patient_split / f"{patient_id}.txt"
                with open(output_file, 'w', encoding='utf-8') as f:
                    f.write(full_prompt)
                
                processed_count += 1
                
            except Exception as e:
                error_count += 1
                error_log.append({
                    'patient_id': patient_id,
                    'condition': condition['condition_name'],
                    'error': str(e)
                })
                
        # Progress update every 50 patients
        if (batch_start + idx + 1) % 50 == 0:
            print(f"  Processed {batch_start + idx + 1}/{total_patients} patients")

print(f"\nProcessing complete!")
print(f"Successfully generated: {processed_count:,} prompts")
print(f"Errors encountered: {error_count:,}")
print(f"Success rate: {processed_count/total_outputs*100:.1f}%")


Processing 22591 patients across 16 conditions
Total outputs to generate: 361,456
Processing in batches of 100


Processing batches:   0%|          | 0/226 [00:00<?, ?it/s]

  Processed 50/22591 patients


Processing batches:   0%|          | 1/226 [00:09<36:55,  9.85s/it]

  Processed 100/22591 patients
  Processed 150/22591 patients


...

  Processed 1900/22591 patients
  Processed 1950/22591 patients


Processing batches:   9%|▉         | 20/226 [07:02<1:40:36, 29.30s/it]

  Processed 2000/22591 patients
  Processed 2050/22591 patients


Processing batches:   9%|▉         | 21/226 [07:35<1:43:32, 30.30s/it]

  Processed 2100/22591 patients


In [ ]:
# Load preprocessed data and objects
import sys
import warnings
warnings.filterwarnings('ignore')

phase1_dir = Path('../Phase 1/phase_1_outputs')

def load_pickle_safe(filepath):
    """Simple pandas compatibility fix for pickle loading"""
    try:
        with open(filepath, 'rb') as f:
            return pickle.load(f)
    except ModuleNotFoundError:
        # Handle pandas version compatibility
        class CompatibilityModule:
            def __getattr__(self, name):
                return pd.Index if name.endswith('Index') else pd.RangeIndex
        
        sys.modules['pandas.core.indexes.numeric'] = CompatibilityModule()
        try:
            with open(filepath, 'rb') as f:
                return pickle.load(f)
        finally:
            if 'pandas.core.indexes.numeric' in sys.modules:
                del sys.modules['pandas.core.indexes.numeric']

# Load data files
label_encoders = load_pickle_safe(phase1_dir / 'preprocessed_mort_hosp_trends_True_window_24_gap_6_seed_42_label_encoders.pkl')
scaler = load_pickle_safe(phase1_dir / 'preprocessed_mort_hosp_trends_True_window_24_gap_6_seed_42_scaler.pkl')
X_test = load_pickle_safe(phase1_dir / 'preprocessed_mort_hosp_trends_True_window_24_gap_6_seed_42_X_test.pkl')

print(f"Dataset shape: {X_test.shape}")
print(f"Total features: {len(scaler.feature_names_in_)}")

print(f"\nCategorical variables ({len(label_encoders)}):")
for var, encoder_dict in label_encoders.items():
    if isinstance(encoder_dict, dict) and 'classes' in encoder_dict:
        classes = encoder_dict['classes']
        print(f"  {var}: {classes}")
    elif hasattr(encoder_dict, 'classes_'):
        print(f"  {var}: {list(encoder_dict.classes_)}")
    else:
        print(f"  {var}: Could not extract classes from {type(encoder_dict)}")


Dataset shape: (5648, 458)
Total features: 458

Categorical variables (3):
  gender_encoded: ['F', 'M']
  ethnicity_encoded: ['AMERICAN INDIAN/ALASKA NATIVE', 'AMERICAN INDIAN/ALASKA NATIVE FEDERALLY RECOGNIZED TRIBE', 'ASIAN', 'ASIAN - ASIAN INDIAN', 'ASIAN - CAMBODIAN', 'ASIAN - CHINESE', 'ASIAN - FILIPINO', 'ASIAN - JAPANESE', 'ASIAN - KOREAN', 'ASIAN - OTHER', 'ASIAN - THAI', 'ASIAN - VIETNAMESE', 'BLACK/AFRICAN', 'BLACK/AFRICAN AMERICAN', 'BLACK/CAPE VERDEAN', 'BLACK/HAITIAN', 'CARIBBEAN ISLAND', 'HISPANIC OR LATINO', 'HISPANIC/LATINO - CENTRAL AMERICAN (OTHER)', 'HISPANIC/LATINO - COLOMBIAN', 'HISPANIC/LATINO - CUBAN', 'HISPANIC/LATINO - DOMINICAN', 'HISPANIC/LATINO - GUATEMALAN', 'HISPANIC/LATINO - HONDURAN', 'HISPANIC/LATINO - MEXICAN', 'HISPANIC/LATINO - PUERTO RICAN', 'HISPANIC/LATINO - SALVADORAN', 'MIDDLE EASTERN', 'MULTI RACE ETHNICITY', 'NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER', 'OTHER', 'PATIENT DECLINED TO ANSWER', 'PORTUGUESE', 'SOUTH AMERICAN', 'UNABLE TO OBTAIN', '

## Generate Summary Statistics


In [ ]:
# Generate summary statistics
summary_stats = {}
total_files = 0
total_size_mb = 0

print("Generating summary statistics...")

for condition in conditions:
    condition_dir = output_base / condition['condition_name']
    condition_stats = {
        'splits': {},
        'total_files': 0,
        'total_size_mb': 0
    }
    
    # Check each split directory
    for split in splits:
        split_dir = condition_dir / split
        files = list(split_dir.glob('*.txt'))
        
        if files:
            # Sample a few files to get size statistics
            sample_files = files[:min(10, len(files))]
            sizes = [f.stat().st_size for f in sample_files]
            
            split_stats = {
                'file_count': len(files),
                'avg_size_bytes': int(np.mean(sizes)),
                'total_size_mb': sum(f.stat().st_size for f in files) / (1024 * 1024)
            }
            
            condition_stats['splits'][split] = split_stats
            condition_stats['total_files'] += len(files)
            condition_stats['total_size_mb'] += split_stats['total_size_mb']
            
            total_files += len(files)
            total_size_mb += split_stats['total_size_mb']
    
    summary_stats[condition['condition_name']] = condition_stats

# Print summary
print(f"\n{'='*60}")
print(f"SERIALIZATION SUMMARY")
print(f"{'='*60}")
print(f"Total files generated: {total_files:,}")
print(f"Total size: {total_size_mb:.1f} MB")
print(f"Output directory: {output_base.absolute()}")

print(f"\nPer-condition breakdown:")
for condition_name, stats in summary_stats.items():
    print(f"  {condition_name}:")
    print(f"    Total files: {stats['total_files']:,}")
    print(f"    Total size: {stats['total_size_mb']:.1f} MB")
    for split, split_stats in stats['splits'].items():
        print(f"      {split}: {split_stats['file_count']:,} files, {split_stats['total_size_mb']:.1f} MB")

# Save summary to JSON
summary_file = output_base / 'generation_summary.json'
with open(summary_file, 'w') as f:
    json.dump({
        'total_files': total_files,
        'total_size_mb': total_size_mb,
        'conditions': summary_stats,
        'error_count': error_count,
        'success_rate': processed_count/total_outputs if total_outputs > 0 else 0
    }, f, indent=2)

print(f"\nSummary saved to: {summary_file}")

# Show next steps
print(f"\n{'='*60}")
print(f"SERIALIZATION COMPLETE")
print(f"{'='*60}")
print(f"Ready for LLM processing!")
print(f"\nNext steps:")
print(f"1. Send prompts to your chosen LLM")
print(f"2. Collect embeddings for each condition")
print(f"3. Train classifiers on embeddings")
print(f"4. Compare performance across conditions")
